# Autonomous Research Assistants
Companion notebook for the **Autonomous Research Assistants** section of Chapter 11.

The goal isn't "find something and write nicely" -- it's a supported answer with clear links between claims and sources. Plan into sub-questions, keep structured notes, separate **claims** from **evidence**, and reject anything factual with no attached source.

A small `Source`/`Claim` shape matching the note fields the book calls for -- source ID, quoted passage, date, sub-question, confidence:

In [1]:
from dataclasses import dataclass, field


@dataclass
class Source:
    source_id: str
    url_or_doc_id: str
    opened: bool  # did the agent actually open/read this, or just guess the URL?


@dataclass
class Note:
    source_id: str
    quoted_passage: str
    date: str
    sub_question: str
    confidence: str  # "high" | "medium" | "low"


@dataclass
class Claim:
    text: str
    is_factual: bool
    source_ids: list[str] = field(default_factory=list)
    sources: list[Source] = field(default_factory=list)

**Planning**: break a broad research task into focused sub-questions -- this keeps search targeted and makes gaps easier to spot:

In [2]:
def plan_subquestions(task: str) -> list[str]:
    """A fixed decomposition for this one task shape, matching the book's own example --
    a real planner would use an LLM here, but the point of this section is the structure
    that comes after planning, not the planning call itself."""
    if "compare vendors" in task.lower():
        return ["pricing", "security", "integrations", "accuracy claims", "customer references"]
    return [task]


subquestions = plan_subquestions("Compare vendors for document processing")
print(subquestions)

['pricing', 'security', 'integrations', 'accuracy claims', 'customer references']


This is the book's `validate_claim`, unchanged:

In [3]:
def validate_claim(claim):
    if claim.is_factual and not claim.source_ids:
        return "reject"
    if any(src.opened is False for src in claim.sources):
        return "reject"
    return "ok"

In [4]:
opened_source = Source(source_id="s1", url_or_doc_id="https://vendor-a.com/soc2", opened=True)
guessed_source = Source(source_id="s2", url_or_doc_id="https://vendor-a.com/made-up-page", opened=False)

supported_claim = Claim(text="Vendor A holds a SOC 2 Type II report.", is_factual=True,
                        source_ids=["s1"], sources=[opened_source])
unsourced_claim = Claim(text="Vendor A is fully HIPAA certified.", is_factual=True, source_ids=[])
guessed_url_claim = Claim(text="Vendor A has zero downtime since 2019.", is_factual=True,
                          source_ids=["s2"], sources=[guessed_source])
opinion = Claim(text="Vendor A's docs are easy to read.", is_factual=False)

for claim in [supported_claim, unsourced_claim, guessed_url_claim, opinion]:
    print(f"{validate_claim(claim):8s} <- {claim.text!r}")

ok       <- 'Vendor A holds a SOC 2 Type II report.'
reject   <- 'Vendor A is fully HIPAA certified.'
reject   <- 'Vendor A has zero downtime since 2019.'
ok       <- "Vendor A's docs are easy to read."


**Conflicts**: don't smooth over disagreeing sources -- mark the conflict instead. A marketing page claiming "enterprise-grade security" and an audit report giving the actual certification scope are both evidence, but they don't say the same thing:

In [5]:
def check_conflict(notes: list[Note]) -> list[str]:
    """Flags sub-questions where notes disagree in confidence -- a simple proxy for
    'these sources don\'t actually agree, don\'t merge them into one confident claim.'"""
    by_subq: dict[str, set[str]] = {}
    for note in notes:
        by_subq.setdefault(note.sub_question, set()).add(note.confidence)
    return [subq for subq, confidences in by_subq.items() if len(confidences) > 1]


notes = [
    Note("s1", "SOC 2 Type II, scope: infrastructure only", "2026-01-10", "security", "high"),
    Note("s2", "marketing page: 'enterprise-grade security'", "2026-02-01", "security", "low"),
    Note("s3", "list pricing $49/seat/mo", "2026-01-15", "pricing", "high"),
]

conflicted = check_conflict(notes)
print("sub-questions with disagreeing evidence:", conflicted)
print("-> summary should state the broader security claim is not fully supported by the audit scope.")

sub-questions with disagreeing evidence: ['security']
-> summary should state the broader security claim is not fully supported by the audit scope.


Even with every claim validated, a research agent's output still needs a **human review step** before use -- it speeds up collection and comparison, it doesn't replace judgment on weak or stale evidence.